# DGA Transformer Fault Diagnosis: Random Forest vs KNN vs SVM vs ANN
Replication of the KNUST preprint on the **DuVal.csv** dataset (ahmedtariq71/Dataset1).

**Run cells top to bottom.** Note: this dataset has 4 classes (Normal, PD, OH, Arcing) and 627 rows, not the paper's 6 IEC classes / 589 rows, so numbers will differ from the paper.

## Step 1: Install and import

In [ ]:
!pip install -q imbalanced-learn
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # applies SMOTE ONLY on training folds
warnings.filterwarnings('ignore')
SEED = 42

## Step 2: Load the data
We load the single full file `DuVal.csv` (not the supplied Train/Test files, see the data-quality check below).

In [ ]:
URL = 'https://raw.githubusercontent.com/ahmedtariq71/Dataset1/main/DuVal.csv'
df = pd.read_csv(URL)
print(df.shape)
display(df.head())
print(df['Fault'].value_counts())

## Step 3: Clean the data (important)
Three problems exist in this dataset:
1. **Duplicate rows** (about 175), which would leak between train and test.
2. **Conflicting labels**: identical gas readings with different fault labels.
3. The provided `DuVal_test.csv` shares about half its rows with `DuVal_Train.csv`, so it is not a valid held-out set.

In [ ]:
GASES = ['H2','CH4','C2H6','C2H4','C2H2']      # paper uses these 5 gases (CO is ignored)
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()
# drop feature vectors that appear with more than one label
conflict = df.groupby(GASES+['CO'])['Fault'].transform('nunique') > 1
print('Conflicting-label rows dropped:', conflict.sum())
df = df[~conflict].reset_index(drop=True)
print(df.shape); print(df['Fault'].value_counts())

## Step 4: Feature engineering (15 features, as in paper Section 4.2)
5 log1p gases + 6 ratios + TCG + 3 Duval percentages = 15.

In [ ]:
def make_features(d):
    e = 1e-6
    g = d[GASES].astype(float)
    X = pd.DataFrame(index=d.index)
    for c in GASES: X[f'log_{c}'] = np.log1p(g[c])
    X['R_CH4_H2']   = g.CH4/(g.H2+e)
    X['R_C2H2_C2H4']= g.C2H2/(g.C2H4+e)
    X['R_C2H4_C2H6']= g.C2H4/(g.C2H6+e)
    X['R_C2H2_CH4'] = g.C2H2/(g.CH4+e)
    X['R_C2H2_H2']  = g.C2H2/(g.H2+e)
    X['R_C2H6_C2H2']= g.C2H6/(g.C2H2+e)
    X['TCG'] = g.sum(axis=1)
    s = g.CH4+g.C2H4+g.C2H2+e
    X['Duval_CH4pct']  = 100*g.CH4/s
    X['Duval_C2H4pct'] = 100*g.C2H4/s
    X['Duval_C2H2pct'] = 100*g.C2H2/s
    return X

X = make_features(df)
classes = sorted(df['Fault'].unique())
y = df['Fault'].map({c:i for i,c in enumerate(classes)}).values
print(X.shape, classes)

## Step 5: Stratified 80/20 split
The test set is untouched until the final evaluation.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print('train', X_tr.shape, np.bincount(y_tr), '| test', X_te.shape, np.bincount(y_te))

## Step 6: Models, SMOTE and hyperparameter grids
SMOTE sits inside each pipeline, so it is fit only on the training part of every CV fold (no leakage). `k_neighbors=1` is used because the Arcing class is tiny.

KNN, SVM and ANN get a `StandardScaler`; Random Forest does not need one.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=SEED)   # 3 folds: Arcing has only a handful of samples
sm = lambda: SMOTE(k_neighbors=1, random_state=SEED)

specs = {
 'Random Forest': (Pipeline([('smote',sm()),('m',RandomForestClassifier(random_state=SEED,n_jobs=-1))]),
    {'m__n_estimators':[50,100,200,300,500],'m__min_samples_leaf':[1,2,5],'m__max_features':['sqrt',None]}),
 'KNN': (Pipeline([('sc',StandardScaler()),('smote',sm()),('m',KNeighborsClassifier())]),
    {'m__n_neighbors':list(range(1,26,2))}),
 'SVM (RBF)': (Pipeline([('sc',StandardScaler()),('smote',sm()),('m',SVC(kernel='rbf',decision_function_shape='ovo'))]),
    {'m__C':[0.1,1,10,100],'m__gamma':['scale',0.01,0.1,1]}),
 'ANN': (Pipeline([('sc',StandardScaler()),('smote',sm()),
    ('m',MLPClassifier(hidden_layer_sizes=(20,10),early_stopping=True,max_iter=2000,random_state=SEED))]),
    {'m__alpha':[1e-4,1e-3,1e-2],'m__learning_rate_init':[1e-3,1e-2]}),
}

## Step 7: Tune with grid search (CV on the training set only)
Takes a few minutes.

In [ ]:
best = {}
for name,(pipe,grid) in specs.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring='accuracy', n_jobs=-1).fit(X_tr, y_tr)
    best[name] = gs
    print(f'{name:14s} CV acc = {gs.best_score_:.4f} | best = {gs.best_params_}')

## Step 8: Evaluate on the held-out test set
Accuracy plus macro precision, recall and F1 (Table 5 in the paper).

In [ ]:
rows, preds = [], {}
for name,gs in best.items():
    p = gs.predict(X_te); preds[name] = p
    pr,rc,f1,_ = precision_recall_fscore_support(y_te,p,average='macro',zero_division=0)
    rows.append([name, accuracy_score(y_te,p)*100, pr*100, rc*100, f1*100, gs.best_score_*100])
res = pd.DataFrame(rows, columns=['Model','Accuracy','Precision','Recall','F1','CV Acc']).round(2)
display(res.sort_values('Accuracy',ascending=False))
for name,p in preds.items():
    print('\n==',name); print(classification_report(y_te,p,target_names=classes,zero_division=0))

## Step 9: Figures
Confusion matrices (Figs 2, 5, 6, 8) and the accuracy and P/R/F1 comparisons (Figs 9, 10).

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(11,9))
for a,(name,p) in zip(ax.ravel(),preds.items()):
    sns.heatmap(confusion_matrix(y_te,p),annot=True,fmt='d',cmap='Blues',cbar=False,
                xticklabels=classes,yticklabels=classes,ax=a)
    a.set_title(f'{name} (Acc = {accuracy_score(y_te,p)*100:.2f}%)'); a.set_xlabel('Predicted'); a.set_ylabel('True')
plt.tight_layout(); plt.show()

res.set_index('Model')[['Accuracy']].plot.bar(legend=False,figsize=(6,4),color='#1f3864',rot=0)
plt.ylabel('Test accuracy (%)'); plt.title('Overall Test Accuracy'); plt.ylim(0,100); plt.show()
res.set_index('Model')[['Precision','Recall','F1']].plot.bar(figsize=(8,4),rot=0)
plt.ylabel('Score (%)'); plt.title('Macro Precision / Recall / F1'); plt.ylim(0,100); plt.show()

## Step 10: Random Forest extras: OOB error curve (Fig 1) and feature importance (Fig 3)

In [ ]:
bp = {k.replace('m__',''):v for k,v in best['Random Forest'].best_params_.items()}
bp.pop('n_estimators',None)
Xs,ys = SMOTE(k_neighbors=1,random_state=SEED).fit_resample(X_tr,y_tr)
rf = RandomForestClassifier(n_estimators=1,warm_start=True,oob_score=True,bootstrap=True,
                            random_state=SEED,n_jobs=-1,**bp)
oob=[]
for n in range(20,501,10):
    rf.set_params(n_estimators=n); rf.fit(Xs,ys); oob.append((n,1-rf.oob_score_))
plt.plot(*zip(*oob)); plt.xlabel('Number of trees'); plt.ylabel('OOB error'); plt.title('Random Forest OOB error'); plt.show()

pi = permutation_importance(best['Random Forest'], X_te, y_te, n_repeats=20, random_state=SEED, scoring='accuracy')
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()
imp.plot.barh(figsize=(7,6)); plt.title('Permutation feature importance (test set)'); plt.show()

## Step 11 (optional): Cross-dataset generalization (paper Section 5.6)
The paper's two independent multi-source datasets are **not** in this repo. `Rogers_Rules*.csv` and `DRM_Rules*.csv` are the same 627 samples converted to ratios, so they are **not independent** and cannot be used for this test. To reproduce Section 5.6 you would need another source, e.g. the IEEE DataPort DGA datasets. Once you have a CSV with the same gas columns and labels:

In [ ]:
# ext = pd.read_csv('your_other_dataset.csv')      # needs H2,CH4,C2H6,C2H4,C2H2 and a 'Fault' column
# ext = ext[ext['Fault'].isin(classes)]             # labels must match: Normal, PD, OH, Arcing
# Xe = make_features(ext); ye = ext['Fault'].map({c:i for i,c in enumerate(classes)})
# print('Cross-dataset accuracy:', accuracy_score(ye, best['Random Forest'].predict(Xe))*100)